# Phase 2: Nepali LMA Model Pretraining & Evaluation (Kaggle)

This notebook trains and evaluates the ~25M parameter Decoder-only Transformer on `manyajain111/nepali-lma-corpus`.

### Setup Instructions:
1. **Accelerator**: Select **GPU (T4 x2 or P100)** in the Kaggle right-sidebar under **Settings -> Accelerator**.
2. **Upload Code**: Upload `phase2` folder or clone your GitHub repository into Kaggle.
3. Run the cells below.

In [ ]:
# Install evaluation dependencies
!pip install -q datasets sacrebleu rouge_score seaborn matplotlib tqdm

In [ ]:
# 1. Run Pretraining (AdamW + Cosine Warmup + Mixed Precision + Checkpointing)
# Token budget: 16 (batch) x 512 (ctx_len) x 61036 (steps) = ~500M tokens
!python phase2/src/train.py \
    --config phase2/config/model_l_config.json \
    --tokenizer phase2/tokenizers/nepali_tokenizer_16000.json \
    --dataset manyajain111/nepali-lma-corpus \
    --output_dir /kaggle/working/nepali_checkpoints \
    --batch_size 16 \
    --grad_accum 4 \
    --max_steps 61036 \
    --save_steps 5000 \
    --val_steps 1000


In [ ]:
# 2. Run Evaluation (PPL, BPB, BLEU, chrF, ROUGE-L, Diversity, Attention Maps/Entropy/Distance)
!python phase2/src/evaluate.py \
    --checkpoint /kaggle/working/nepali_checkpoints/model_final.pt \
    --tokenizer phase2/tokenizers/nepali_tokenizer_16000.json \
    --dataset manyajain111/nepali-lma-corpus \
    --language nepali \
    --output_dir /kaggle/working/nepali_eval_reports